# Self-supervised learning: making a vector space organise itself

Every lesson so far needed a label. Lesson 2 froze someone else's encoder and trained a head on
top; lesson 5 did the same for text. But *someone* had to train those encoders, and the interesting
ones were not trained on labels — there are not enough labels in the world.

This notebook builds that "someone else" machinery from scratch, one piece at a time, and ends with
a vector space that got measurably more organised **without ever seeing a label during training**.

The whole family of methods — SimCLR, MoCo, BYOL, SimSiam, DINO, I-JEPA — is one block with four
switches. We build the block, then flip the switches.

| axis | values | who does what |
|---|---|---|
| **views** | augment twice / mask one | SimCLR, MoCo, BYOL / I-JEPA |
| **target weights** | shared (`m=0`) / EMA copy (`m→1`) | SimSiam / MoCo, BYOL, DINO |
| **head** | none / predictor | MoCo, SimCLR / BYOL, SimSiam, I-JEPA |
| **objective** | InfoNCE (negatives) / cosine + stop-grad | MoCo, SimCLR / BYOL, SimSiam |

> **Runtime**: about **15 minutes** end to end on a CPU laptop (the five-config comparison in
> section 5 is ~10 of those). Every training run here is deliberately short — we are looking for
> *direction*, not for a competitive model.

## The dataset, and why this one

**EuroSAT**: 64x64 RGB satellite tiles, 10 land-use classes (`Forest`, `River`, `Residential`,
`Highway`, ...). We use it for three reasons:

1. **There is room to show something.** A linear probe on the raw pixels scores **0.335**. Land use
   is *texture*, and texture is not linearly readable off pixel values, so most of the distance to
   1.0 is still there to be won. Compare MNIST, where raw pixels already score **0.869** — there any
   encoder looks impressive, because there is almost nothing left to win.
2. **Rotation is a real invariance here.** A field rotated 30 degrees is still a field. That is not
   true of a handwritten `2`, and it is very much not true of a shirt (see section 7).
3. **It is the actual problem.** Aerial imagery is where self-supervised learning earns its keep in
   practice: enormous amounts of unlabelled imagery, expensive labels.

In [ ]:
import time

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch import Tensor, nn
from torchvision.transforms import v2

torch.manual_seed(42)

RES = 32          # downscale 64->32; everything here is about direction, not absolute score
DIM = 128         # embedding width
BATCH = 256
N_TRAIN, N_TEST = 8_000, 2_000

## 1. Two views — the only genuinely creative decision

Self-supervised learning needs a signal, and with no labels there is only one place to get it: we
*declare* that two different-looking things are the same thing, and make the model agree.

That declaration is the augmentation. **Choosing augmentations is choosing what the model is allowed
to throw away.** It is the whole ballgame, and section 7 shows what happens when you choose wrong.

In [ ]:
from datasets import load_dataset

ds = load_dataset("blanchon/EuroSAT_RGB")
CLASSES = ds["train"].features["label"].names
print(f"{len(ds['train'])} train tiles, {len(CLASSES)} classes: {CLASSES}")

to_tensor = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Resize((RES, RES), antialias=True),
])


def pack(split, n, seed=0):
    """HF dataset -> one big (n, 3, RES, RES) float tensor. Small enough to hold in RAM."""
    d = split.shuffle(seed=seed)
    X = torch.stack([to_tensor(d[i]["image"].convert("RGB")) for i in range(n)])
    y = torch.tensor([d[i]["label"] for i in range(n)])
    return X, y


t0 = time.time()
Xtr, ytr = pack(ds["train"], N_TRAIN)
Xte, yte = pack(ds["test"], N_TEST)
print(f"packed {tuple(Xtr.shape)} train, {tuple(Xte.shape)} test in {time.time()-t0:.0f}s")

> **A trap this notebook walked into first.** EuroSAT's test split is **ordered by class**: the first
> 2.000 rows contain only 4 of the 10 classes. `pack` shuffles *before* slicing for exactly that
> reason. An earlier draft did not, and every number it produced was scored against a 4-class test
> set — which made the raw-pixel floor look far lower (0.157) and the trained model far better
> (0.842) than either really was. Nothing errored; the shapes were all correct.
>
> Whenever you slice a split, check what is actually in your slice.

In [ ]:
# The augmentation. Note what is NOT here: no horizontal flip would be wrong for text or digits,
# but for aerial tiles every orientation is legitimate, so we rotate freely.
augment = v2.Compose([
    v2.RandomResizedCrop(RES, scale=(0.4, 1.0), antialias=True),
    v2.RandomHorizontalFlip(),
    v2.RandomRotation((-20.0, 20.0)),
    v2.RandomApply([v2.ColorJitter(brightness=0.4, contrast=0.4)], p=0.8),
])

i = 7
tile = Xtr[i : i + 1]
fig, axes = plt.subplots(1, 5, figsize=(12, 2.6))
axes[0].imshow(tile[0].permute(1, 2, 0)); axes[0].set_title(f"original\n({CLASSES[ytr[i]]})")
for ax in axes[1:]:
    ax.imshow(augment(tile)[0].permute(1, 2, 0)); ax.set_title("a 'view'")
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

Look at those four views and ask the question the rest of the notebook answers:

> **Why should all of these produce the same vector?**

Nothing forces it. We are about to *impose* it, and the imposing is what does the learning.

## 2. The naive idea, and the trap it walks into

Simplest possible plan: push two views through one encoder, and maximise the cosine similarity
between the outputs. Let us just try it.

In [ ]:
class Encoder(nn.Module):
    """A small conv net. Nothing clever - the lesson is in the training signal, not the architecture."""

    def __init__(self, in_ch: int = 3, dim: int = DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, dim),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)


def batches(X, steps, batch=BATCH):
    """Two independently augmented views of the same random batch, `steps` times."""
    for _ in range(steps):
        b = X[torch.randint(0, len(X), (batch,))]
        yield augment(b), augment(b)

In [ ]:
from notebooktester import param
torch.manual_seed(42)
naive = Encoder()
opt = torch.optim.Adam(naive.parameters(), lr=1e-3)

losses = []
STEPS = param(200, test=2)

for view_a, view_b in batches(Xtr, steps=STEPS):
    opt.zero_grad()
    z1, z2 = naive(view_a), naive(view_b)
    loss = -F.cosine_similarity(z1, z2, dim=-1).mean()   # "make the two views agree"
    loss.backward()
    opt.step()
    losses.append(loss.item())

plt.figure(figsize=(6, 3))
plt.plot(losses); plt.xlabel("step"); plt.ylabel("loss"); plt.title("naive: loss")
plt.tight_layout(); plt.show()
print(f"final loss {losses[-1]:.4f}   (theoretical minimum is -1.0)")

The loss went almost exactly to **-1.0**, its theoretical minimum. Perfect agreement between the two
views. By the only number we have been watching, this is a total success.

It is a total failure. There is a trivial way to make any two views agree: **output the same vector
for every input.** The model found it in about thirty seconds.

This is *representation collapse*, and it is the central failure mode of the whole field. Notice that
the loss curve cannot tell you it happened — the loss is delighted.

## 3. An instrument that can see collapse

We need a second number. If every input maps to the same vector, then across a batch each dimension
of the (normalised) embedding has **zero variance**. So: normalise, take the standard deviation per
dimension, average.

A healthy `DIM`-dimensional representation spread over the unit sphere sits near `1/sqrt(DIM)`.
For `DIM=128` that is **0.088**. Collapsed is **0.000**.

In [ ]:
def rep_std(z: Tensor) -> float:
    """Collapse detector: mean per-dimension std of L2-normalised embeddings.

    ~0.000        -> collapsed, every input maps to the same vector
    ~1/sqrt(DIM)  -> healthy, the batch is spread over the sphere
    """
    return F.normalize(z, dim=-1).std(dim=0).mean().item()


print(f"healthy target for DIM={DIM}: {1/DIM**0.5:.4f}")
print(f"our naive encoder:            {rep_std(naive(Xte[:512])):.4f}   <- collapsed")

That is the instrument. From here on **every run reports loss *and* representation std**, because
loss alone is not evidence.

## 4. The block: `Parallel`, with the weights coupled

In lesson 5 you met `vectormesh.Parallel` — run several branches over several inputs, return a tuple.
Two independent branches, two independent sets of weights.

The self-supervised block is `Parallel` with exactly two changes:

1. **The weights are coupled.** Either literally the same module (`momentum=0`), or a slowly-trailing
   copy updated by `theta_target <- m * theta_target + (1-m) * theta_online` (`momentum>0`).
2. **One branch has a gradient gate.** The target side is wrapped in `no_grad`, so it produces a
   *fixed goal* for this step rather than something that can move to meet the online branch halfway.

That second point is the fix for section 2. Collapse is the two branches conspiring; the stop-gradient
means only one of them is allowed to move.

In [ ]:
class Siamese(nn.Module):
    """Two coupled branches over two views. `Parallel`, plus weight-coupling and a gradient gate.

    momentum = 0.0   -> the target IS the online encoder (shared weights).  SimSiam
    momentum > 0.0   -> the target is an EMA copy that trails behind.       MoCo, BYOL, DINO
    predictor        -> an extra head on the online side only.              BYOL, SimSiam, I-JEPA
    stopgrad = False -> the ablation from section 2. Expect collapse.
    """

    def __init__(self, online: nn.Module, momentum: float = 0.0,
                 predictor: nn.Module | None = None, stopgrad: bool = True):
        super().__init__()
        self.online, self.momentum, self.stopgrad = online, momentum, stopgrad
        self.predictor = predictor
        if momentum > 0.0:
            import copy
            self.target = copy.deepcopy(online)
            for p in self.target.parameters():
                p.requires_grad = False   # the target is never trained, only averaged into
        else:
            self.target = None

    @torch.no_grad()
    def update_target(self) -> None:
        """The EMA step. Lives here (called from forward) rather than in the training loop,
        so this block drops into any trainer without that trainer knowing SSL exists."""
        if self.target is None:
            return
        m = self.momentum
        for tp, op in zip(self.target.parameters(), self.online.parameters()):
            tp.mul_(m).add_(op.detach(), alpha=1 - m)
        for tb, ob in zip(self.target.buffers(), self.online.buffers()):
            tb.copy_(ob)                  # BatchNorm running stats: copy, don't average

    def _one_way(self, v_online: Tensor, v_target: Tensor) -> tuple[Tensor, Tensor]:
        z_on = self.online(v_online)
        target = self.target if self.target is not None else self.online
        if self.stopgrad:
            with torch.no_grad():         # <- the gradient gate
                z_tg = target(v_target)
        else:
            z_tg = target(v_target)
        p = self.predictor(z_on) if self.predictor is not None else z_on
        return p, z_tg

    def forward(self, views: tuple[Tensor, Tensor]):
        view_a, view_b = views
        if self.training:
            self.update_target()
        # symmetrised: each view takes a turn as the target
        return self._one_way(view_a, view_b), self._one_way(view_b, view_a)


def predictor_head(dim: int = DIM, hidden: int = 512) -> nn.Module:
    return nn.Sequential(
        nn.Linear(dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Linear(hidden, dim)
    )

### The two objectives

**Cosine + stop-grad** (BYOL, SimSiam): pull the two views together, and rely on the stop-gradient and
the predictor to stop everything piling up in one place. No negatives at all.

**InfoNCE** (SimCLR, MoCo): pull the two views of *this* tile together **and push away every other tile
in the batch**. The pushing-away is an explicit anti-collapse force, which is why this branch is the
robust one at small budgets.

In [ ]:
def cosine_loss(out) -> Tensor:
    """BYOL / SimSiam. Only positives."""
    (p1, z2), (p2, z1) = out
    return -(F.cosine_similarity(p1, z2, dim=-1).mean()
             + F.cosine_similarity(p2, z1, dim=-1).mean()) / 2


def infonce_loss(out, temp: float = 0.2) -> Tensor:
    """SimCLR / MoCo. The other tiles in the batch are the negatives.

    Classify 'which of the N tiles in this batch is my partner?' -> plain cross-entropy
    over a similarity matrix. MoCo's queue is this, with negatives kept from earlier batches.
    """
    def half(p, z):
        p, z = F.normalize(p, dim=-1), F.normalize(z, dim=-1)
        return F.cross_entropy(p @ z.t() / temp, torch.arange(len(p)))

    (p1, z2), (p2, z1) = out
    return (half(p1, z2) + half(p2, z1)) / 2

## 5. One block, five papers

Now the table from the top becomes five configurations of the same object.

In [ ]:
STEPS = param(300, test=2)

def train_ssl(name, momentum=0.0, use_predictor=False, stopgrad=True,
              loss_fn=cosine_loss, steps=STEPS, seed=42, verbose=True):
    torch.manual_seed(seed)
    encoder = Encoder()
    model = Siamese(encoder, momentum,
                    predictor_head() if use_predictor else None, stopgrad)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    model.train()

    t0, hist = time.time(), []
    for step, (view_a, view_b) in enumerate(batches(Xtr, steps)):
        opt.zero_grad()
        out = model((view_a, view_b))
        loss = loss_fn(out)
        loss.backward()
        opt.step()
        if step % 25 == 0:
            hist.append((step, loss.item(), rep_std(out[0][0].detach())))
    if verbose:
        print(f"{name:22s} loss {loss.item():+7.3f}  repstd {hist[-1][2]:.4f}  [{time.time()-t0:.0f}s]")
    return encoder, hist


configs = {
    "SimSiam":              dict(momentum=0.0,  use_predictor=True,  stopgrad=True,  loss_fn=cosine_loss),
    "BYOL (EMA target)":    dict(momentum=0.99, use_predictor=True,  stopgrad=True,  loss_fn=cosine_loss),
    "InfoNCE + EMA target": dict(momentum=0.99, use_predictor=False, stopgrad=True,  loss_fn=infonce_loss),
    "-- no stop-gradient":  dict(momentum=0.0,  use_predictor=True,  stopgrad=False, loss_fn=cosine_loss),
    "-- no predictor":      dict(momentum=0.0,  use_predictor=False, stopgrad=True,  loss_fn=cosine_loss),
}

encoders, histories = {}, {}
for name, cfg in configs.items():
    encoders[name], histories[name] = train_ssl(name, **cfg)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for name, hist in histories.items():
    steps_, loss_, std_ = zip(*hist)
    style = "--" if name.startswith("--") else "-"
    ax1.plot(steps_, loss_, style, label=name)
    ax2.plot(steps_, std_, style, label=name)
ax1.set_title("loss"); ax1.set_xlabel("step")
ax2.axhline(1 / DIM**0.5, color="k", lw=0.8, ls=":", label=f"healthy ({1/DIM**0.5:.3f})")
ax2.set_title("representation std"); ax2.set_xlabel("step"); ax2.set_ylim(bottom=0)
ax2.legend(fontsize=8, loc="center right")
plt.tight_layout(); plt.show()

Read the two panels together — that is the entire point of the section.

- The two dashed ablations reach the **best** loss and the **worst** representation. Removing the
  stop-gradient, or removing the predictor, collapses the space. The loss curve applauds.
- The three real methods keep their representation std up near the healthy line.

The InfoNCE row is the sturdiest of the three at this budget, because its negatives are an *explicit*
repulsive force rather than an emergent one.

**Is that row MoCo?** Almost, and the gap is worth naming. It has InfoNCE and a momentum target
encoder, but its negatives are the other 255 tiles *in the current batch* — which is SimCLR's scheme.
MoCo's actual contribution is the **queue**: a rolling buffer of target embeddings from previous
batches, so the number of negatives stops being tied to the batch size. That mattered enormously when
your batch was limited by GPU memory. Building it is exercise E6.

> A caveat worth carrying: the cosine-based methods (BYOL, SimSiam) are known to be sensitive to
> training length, learning-rate schedule and predictor design. Three hundred steps on a laptop is not
> a fair test of them. Read the dashed lines as "these break", and the solid lines as "these work,
> and InfoNCE is the one that works *easily*".

## 6. Did the space actually get organised?

Time to evaluate, using the protocol this course has used since lesson 1: **freeze the encoder, train a
linear probe on top.** If a linear model can read the classes off the frozen vectors, the vectors are
organised.

Two floors to beat, and the first one matters more than it looks:

- **raw pixels** — a linear probe straight on the flattened image. If we cannot beat this, we have
  achieved literally nothing.
- **a random untrained encoder** — surprisingly strong, because random convolutions already compute
  texture-ish features. This is the floor that says whether *learning* helped, as opposed to architecture.

In [ ]:
def embed(encoder, X, batch=512):
    encoder.eval()
    with torch.no_grad():
        return torch.cat([encoder(X[i : i + batch]) for i in range(0, len(X), batch)])


def linear_probe(Etr, ytr, Ete, yte, epochs=200):
    mu, sd = Etr.mean(0, keepdim=True), Etr.std(0, keepdim=True) + 1e-6
    Etr, Ete = (Etr - mu) / sd, (Ete - mu) / sd
    clf = nn.Linear(Etr.shape[1], int(ytr.max()) + 1)
    opt = torch.optim.Adam(clf.parameters(), lr=1e-2, weight_decay=1e-4)
    for _ in range(epochs):
        opt.zero_grad(); F.cross_entropy(clf(Etr), ytr).backward(); opt.step()
    with torch.no_grad():
        return (clf(Ete).argmax(1) == yte).float().mean().item()


def knn_probe(Etr, ytr, Ete, yte, k=20):
    sim = F.normalize(Ete, dim=-1) @ F.normalize(Etr, dim=-1).t()
    return (ytr[sim.topk(k, 1).indices].mode(1).values == yte).float().mean().item()


torch.manual_seed(42)
rows = [("raw pixels (floor)", linear_probe(Xtr.flatten(1), ytr, Xte.flatten(1), yte), float("nan"), float("nan"))]
rnd = Encoder()
Ertr, Erte = embed(rnd, Xtr), embed(rnd, Xte)
rows.append(("random encoder (floor)", linear_probe(Ertr, ytr, Erte, yte),
             knn_probe(Ertr, ytr, Erte, yte), rep_std(Ertr)))
for name, enc in encoders.items():
    Etr_, Ete_ = embed(enc, Xtr), embed(enc, Xte)
    rows.append((name, linear_probe(Etr_, ytr, Ete_, yte), knn_probe(Etr_, ytr, Ete_, yte), rep_std(Etr_)))

print(f"{'':24s} {'probe':>7s} {'kNN':>7s} {'repstd':>7s}")
print("-" * 50)
for name, p, k, s in rows:
    print(f"{name:24s} {p:7.3f} {k:7.3f} {s:7.4f}")

In [ ]:
# What "organised" looks like. Project the frozen vectors to 2D and colour by a label the
# encoder never saw.
best = "InfoNCE + EMA target"
E = embed(encoders[best], Xte)
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, (title, V) in zip(axes, [("random encoder", embed(rnd, Xte)), (best, E)]):
    Z = torch.pca_lowrank(V - V.mean(0), q=2)[0]
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=yte, cmap="tab10", s=6, alpha=0.6)
    ax.set_title(f"{title}"); ax.set_xticks([]); ax.set_yticks([])
handles = [plt.Line2D([], [], marker="o", ls="", color=plt.cm.tab10(i / 9), label=c)
           for i, c in enumerate(CLASSES)]
fig.legend(handles=handles, loc="center right", fontsize=8)
plt.tight_layout(rect=(0, 0, 0.84, 1)); plt.show()

**That gap is the lesson.** No label was used to train the encoder, and the classes have arranged
themselves anyway — because "these two crops came from the same tile" carries an enormous amount of
information about what is in the tile, and 300 steps of exploiting it is enough to see the structure.

Notice also how far the *raw pixel* floor is below everything else. On MNIST that floor sits at 0.869
and this entire notebook would have looked like a rounding error. Picking a dataset where the floor is
low is not cheating — it is the only way to see whether your method does anything.

## 7. The counter-example: when the augmentation destroys the task

Everything above rests on one assumption: *the augmentations preserve what matters.* Rotating a field
keeps it a field.

So let us break it deliberately. **Fashion-MNIST**: shirt, pullover, coat, bag. What distinguishes them
is silhouette, aspect ratio and orientation — exactly what `RandomResizedCrop` and `RandomRotation`
are designed to make the model ignore.

In [ ]:
from torchvision import datasets

fm_tr = datasets.FashionMNIST(root="data", train=True, download=True)
fm_te = datasets.FashionMNIST(root="data", train=False, download=True)
rs = v2.Resize((RES, RES), antialias=True)
FXtr = rs(fm_tr.data[:N_TRAIN].float().unsqueeze(1) / 255.0)
FXte = rs(fm_te.data[:N_TEST].float().unsqueeze(1) / 255.0)
Fytr, Fyte = fm_tr.targets[:N_TRAIN], fm_te.targets[:N_TEST]

torch.manual_seed(42)
fenc = Encoder(in_ch=1)
fmodel = Siamese(fenc, momentum=0.99, predictor=None, stopgrad=True)
fopt = torch.optim.Adam(fmodel.parameters(), lr=1e-3)
fmodel.train()
for _ in range(STEPS):
    b = FXtr[torch.randint(0, len(FXtr), (BATCH,))]
    fopt.zero_grad()
    loss = infonce_loss(fmodel((augment(b), augment(b))))
    loss.backward(); fopt.step()

FEtr, FEte = embed(fenc, FXtr), embed(fenc, FXte)
print(f"{'raw pixels (floor)':24s} {linear_probe(FXtr.flatten(1), Fytr, FXte.flatten(1), Fyte):7.3f}")
print(f"{'MoCo on fashion':24s} {linear_probe(FEtr, Fytr, FEte, Fyte):7.3f}"
      f"   repstd {rep_std(FEtr):.4f}")

**The self-supervised encoder is worse than logistic regression on raw pixels.**

And note the representation std is *healthy* — it did not collapse. The optimisation worked perfectly.
It successfully learned to be invariant to crop, rotation and flip, and being invariant to those things
is precisely what makes a coat indistinguishable from a pullover.

This is the `designed vs learned` thread of the course, pointed at augmentation:

> The augmentation is not a preprocessing detail. It is where you inject your knowledge of the problem,
> and it is the one part of a self-supervised pipeline no amount of compute can fix.

Aerial imagery is unusually forgiving here — that is a genuine property of the domain, not luck.

## 8. Dropping the block into `mltrainer` without changing `mltrainer`

The hand-written loops above made the mechanism visible. Now the same block, in the trainer the rest
of the course uses. Three things make it fit with no library changes:

1. `Siamese.forward` takes a **tuple** `(v1, v2)`, and `Trainer._to_device` already handles tuple inputs.
2. The EMA update happens **inside `forward`**, so the trainer needs no per-step hook (it has none).
3. `loss_fn(yhat, y)` — our loss simply ignores `y`, and the dataloader hands it a dummy.

And representation std becomes a **metric**, so it is logged next to the loss every epoch instead of
living in a plot we have to remember to look at.

In [ ]:
from mltrainer import ReportTypes, Trainer, TrainerSettings


class RepresentationStd:
    """An mltrainer metric. Called as metric(y, yhat); y is ignored."""
    def __repr__(self):
        return "repstd"

    def __call__(self, y, yhat):
        return rep_std(yhat[0][0])


class TwoViewDataset(torch.utils.data.Dataset):
    """Yields ((view1, view2), dummy_label) so a standard DataLoader can feed SSL."""
    def __init__(self, X):
        self.X = X

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        x = self.X[index : index + 1]
        return (augment(x)[0], augment(x)[0]), 0


loader = torch.utils.data.DataLoader(TwoViewDataset(Xtr), batch_size=BATCH, shuffle=True, drop_last=True)

torch.manual_seed(42)
model = Siamese(Encoder(), momentum=0.99, predictor=None, stopgrad=True)
settings = TrainerSettings(
    epochs=2,
    metrics=[RepresentationStd()],
    logdir="modellogs/ssl",
    train_steps=50,
    valid_steps=10,
    reporttypes=[],
    earlystop_kwargs=None,
    scheduler_kwargs=None,
)
trainer = Trainer(
    model=model, settings=settings, loss_fn=lambda yhat, y: infonce_loss(yhat),
    optimizer=torch.optim.Adam, traindataloader=loader, validdataloader=loader, scheduler=None,
)
trainer.loop()

No subclass, no callback, no fork. The block carries its own update rule, so a trainer that knows
nothing about self-supervised learning trains it correctly — and `repstd` appears in the epoch report,
which means it also flows into MLflow when you switch the report types on.

That is the argument for this living in `vectormesh` as a **component** rather than as training
machinery: composition stays in the model, and the trainer stays dumb.

## What would move into vectormesh

Nothing here is EuroSAT-specific. Written out, the reusable pieces are:

| built here | would become |
|---|---|
| `Siamese` | `vectormesh.components.pipelines.Siamese` — `Parallel` with coupled weights + a gradient gate |
| `rep_std` | `vectormesh.components.metrics.RepresentationStd` — collapse is a metric, not a story |
| `infonce_loss`, `cosine_loss` | `vectormesh.components.losses` |
| `TwoViewDataset` | a `MultiViewCollate`, next to the existing `CollateParallel` |
| — | an adapter wrapping a trained encoder as a `BaseVectorizer`, so it can fill a `VectorCache` |

That last row is the loop closing: this course has spent five lessons *consuming* frozen encoders from
a cache. You have now trained one. The next step is publishing it so somebody else can consume yours.

## Exercises

**E1 — Pick the augmentation, predict the outcome.** Before running anything: which single augmentation
in `augment` do you think matters most on EuroSAT? Remove it, retrain, and check the probe. Were you right?

**E2 — Fix Fashion-MNIST.** Section 7 fails because the augmentations destroy silhouette. Design an
augmentation pipeline that beats the 0.845 raw-pixel floor. (Hint: what is safe to vary about a coat?)

**E3 — Find the momentum boundary.** `MoCo (InfoNCE)` uses `momentum=0.99`. Sweep it: 0.0, 0.9, 0.99,
0.999, 1.0. What happens at exactly 1.0, and why is that not a bug?

**E4 — How little data?** Retrain with 500, 2000, 8000 tiles. Plot the probe score. Self-supervised
learning is supposed to be the thing you do when labels are scarce but *images* are plentiful — does
the curve support that claim?

**E6 — Turn it into real MoCo.** Add a queue: a `register_buffer` ring of the last few thousand
target embeddings, concatenated onto the negatives in `infonce_loss`. Does a 4096-entry queue beat
255 in-batch negatives at this scale? (Think about *why* the answer might be "no here, yes on
ImageNet" before you run it.)

**E5 — The honest comparison.** Train a *supervised* encoder on the same 8000 tiles with labels, and
probe it. How much did the labels buy? Now use only 100 labels for the supervised model, and probe
your self-supervised encoder against it. Where is the crossover?